# Ollama qua REST API thô

`llmkit.chat` gói gọn một lời gọi Ollama trong một dòng. Bài này mở gói đó ra: bạn tự dựng payload `/api/chat`,
đọc **stream NDJSON** từng mảnh, tính tốc độ từ các trường thời gian Ollama trả về, đọc thông tin model
(context length, quantization, giấy phép) từ `/api/show`, viết Modelfile, rồi đo trên server thật: thời gian tới
token đầu (TTFT), token/s, và cái bẫy **`num_ctx`** cắt prompt dài mà không báo lỗi.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell `explore` gọi Ollama thật (cần `ollama serve` với `qwen2.5:3b`), cần bộ `wikipedia-vi` cho prompt dài.
  Server có thể dùng chung: các cell chỉ gọi khoảng 8 lần.
- Chấm: `uv run pytest tracks/ai-engineer/modules/09-llms-and-open-models/exercises/intermediate/02-ollama-rest -q`

In [ ]:
import json
from collections.abc import Iterable

## 1. Payload và stream

`chat_payload(model, messages, *, stream=True, temperature=None, top_p=None, top_k=None, num_ctx=None,
num_predict=None, seed=None, keep_alive=None, format=None) -> dict`:

- Luôn có `model`, `messages`, `stream`.
- Các tham số sampling và `num_ctx`, `num_predict`, `seed` khác `None` nằm trong dict con `options`, đúng tên
  tham số. Không có option nào thì **không** có khoá `options`.
- `keep_alive` (vd. `"5m"`, `0`) và `format` (`"json"` hoặc một JSON schema) khác `None` thì ở cấp ngoài cùng.

`parse_stream(lines) -> tuple[str, dict]`: `lines` là các dòng NDJSON (kiểu `str` hoặc `bytes`) của
`POST /api/chat` khi `stream: true`. Bỏ dòng rỗng. Nối mọi `message.content` thành câu trả lời; trả kèm object
cuối có `"done": true` (chứa thống kê). Dòng có khoá `"error"`: `RuntimeError` mang thông điệp lỗi. Hết dòng mà
chưa gặp `done: true`: `RuntimeError`.

In [ ]:
def chat_payload(model: str, messages: list[dict], *, stream: bool = True, temperature: float | None = None,
                 top_p: float | None = None, top_k: int | None = None, num_ctx: int | None = None,
                 num_predict: int | None = None, seed: int | None = None, keep_alive: str | int | None = None,
                 format: str | dict | None = None) -> dict:
    """Body JSON cho POST /api/chat của Ollama."""
    payload = {"model": model, "messages": messages, "stream": stream}
    options = {"temperature": temperature, "top_p": top_p, "top_k": top_k, "num_ctx": num_ctx,
               "num_predict": num_predict, "seed": seed}
    options = {k: v for k, v in options.items() if v is not None}
    if options:
        payload["options"] = options
    if keep_alive is not None:
        payload["keep_alive"] = keep_alive
    if format is not None:
        payload["format"] = format
    return payload


def parse_stream(lines: Iterable[str | bytes]) -> tuple[str, dict]:
    """Nối content của các mảnh NDJSON; trả (văn bản, object cuối có done=true)."""
    parts: list[str] = []
    for line in lines:
        if isinstance(line, bytes):
            line = line.decode("utf-8")
        if not line.strip():
            continue
        obj = json.loads(line)
        if "error" in obj:
            raise RuntimeError(f"Ollama báo lỗi: {obj['error']}")
        parts.append(obj.get("message", {}).get("content", ""))
        if obj.get("done"):
            return "".join(parts), obj
    raise RuntimeError("stream kết thúc mà chưa có object done=true")

## 2. Tốc độ, thông tin model, Modelfile

`speed_stats(final) -> dict` từ object cuối của stream (mọi thời gian Ollama trả tính bằng **nano giây**):

| Khoá | Cách tính |
| --- | --- |
| `load_s` | `load_duration` (giây) |
| `prompt_tokens`, `gen_tokens` | `prompt_eval_count`, `eval_count` |
| `prompt_tps`, `gen_tps` | số token chia thời gian tương ứng (`prompt_eval_duration`, `eval_duration`) tính bằng giây |
| `total_s` | `total_duration` (giây) |

Trường thiếu coi là 0 (Ollama bỏ `prompt_eval_count` khi prompt đã nằm sẵn trong cache); thời gian bằng 0 thì
tốc độ là `0.0`.

`model_facts(show) -> dict` từ JSON của `POST /api/show`: `architecture` (`model_info["general.architecture"]`),
`parameter_count` (`general.parameter_count`), `context_length` (`model_info["<architecture>.context_length"]`),
`quantization` (`details["quantization_level"]`), `license` (`general.license`, thiếu thì `None`).

`render_modelfile(base, system=None, parameters=None) -> str`: dòng `FROM <base>`, rồi mỗi tham số một dòng
`PARAMETER <tên> <giá trị>` theo thứ tự dict (giá trị là list thì mỗi phần tử một dòng, như `stop`), rồi nếu có
system: `SYSTEM """<system>"""`. Mỗi dòng kết thúc bằng `\n`.

In [ ]:
def speed_stats(final: dict) -> dict:
    """Tốc độ và thời gian từ object done=true của Ollama (ns -> s)."""
    ns = 1e9

    def rate(count: int, duration_ns: int) -> float:
        return count / (duration_ns / ns) if duration_ns else 0.0

    p_tok, g_tok = final.get("prompt_eval_count", 0), final.get("eval_count", 0)
    return {
        "load_s": final.get("load_duration", 0) / ns,
        "prompt_tokens": p_tok,
        "prompt_tps": rate(p_tok, final.get("prompt_eval_duration", 0)),
        "gen_tokens": g_tok,
        "gen_tps": rate(g_tok, final.get("eval_duration", 0)),
        "total_s": final.get("total_duration", 0) / ns,
    }


def model_facts(show: dict) -> dict:
    """Kiến trúc, số tham số, context length, quantization, giấy phép từ /api/show."""
    info = show["model_info"]
    arch = info["general.architecture"]
    return {
        "architecture": arch,
        "parameter_count": info["general.parameter_count"],
        "context_length": info[f"{arch}.context_length"],
        "quantization": show["details"]["quantization_level"],
        "license": info.get("general.license"),
    }


def render_modelfile(base: str, system: str | None = None, parameters: dict | None = None) -> str:
    """Modelfile: FROM, các PARAMETER, SYSTEM."""
    lines = [f"FROM {base}"]
    for name, value in (parameters or {}).items():
        for v in value if isinstance(value, list) else [value]:
            lines.append(f"PARAMETER {name} {v}")
    if system is not None:
        lines.append(f'SYSTEM """{system}"""')
    return "".join(line + "\n" for line in lines)

In [ ]:
# explore: server có gì: model đã tải (/api/tags), model đang nạp trong RAM/VRAM (/api/ps), thông tin từng model
import os
import sys
import time
from pathlib import Path

import httpx

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import llmkit
from data.download_dataset import ensure

HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODEL = "qwen2.5:3b"
http = httpx.Client(base_url=HOST, timeout=1800)
print("Ollama", http.get("/api/version").json()["version"])
for m in http.get("/api/tags").json()["models"]:
    print(f"  {m['name']:24s} {m['size'] / 1e9:5.2f} GB  {m['details']['parameter_size']:>7s}  "
          f"{m['details']['quantization_level']}")
for name in ("qwen2.5:3b", "qwen2.5:7b"):
    try:
        print(name, model_facts(http.post("/api/show", json={"model": name}).json()))
    except KeyError:
        print(name, "chưa pull")

In [ ]:
# explore: stream một câu trả lời, đo thời gian tới token đầu (TTFT) và token/s
messages = [{"role": "user", "content": "Giải thích ngắn gọn trong 3 câu: knowledge cut-off của LLM là gì?"}]
payload = chat_payload(MODEL, messages, temperature=0, seed=42, num_predict=120)
t0, ttft, lines = time.perf_counter(), None, []
with http.stream("POST", "/api/chat", json=payload) as response:
    for line in response.iter_lines():
        if ttft is None and line.strip() and json.loads(line).get("message", {}).get("content"):
            ttft = time.perf_counter() - t0
        lines.append(line)
wall = time.perf_counter() - t0
text, final = parse_stream(lines)
stats = speed_stats(final)
print(text)
print(f"\n{len(lines)} dòng NDJSON, TTFT {ttft:.2f} s, tổng {wall:.2f} s")
print({k: round(v, 2) for k, v in stats.items()})
ps = http.get("/api/ps").json()["models"]
for m in ps:
    print(f"/api/ps: {m['name']} nạp {m['size'] / 1e9:.2f} GB, trên GPU {m['size_vram'] / 1e9:.2f} GB, "
          f"context_length {m.get('context_length')}")

In [ ]:
# explore: cùng câu hỏi qua llmkit (không stream) và endpoint tương thích OpenAI của chính Ollama
same = llmkit.chat(messages, model=MODEL, temperature=0)
print("llmkit, temperature 0 (không seed, không giới hạn num_predict), 200 ký tự đầu giống stream:", same[:200] == text[:200])
oa = http.post("/v1/chat/completions", json={"model": MODEL, "messages": messages, "temperature": 0, "seed": 42,
                                             "max_tokens": 120}).json()
print("OpenAI-compatible /v1/chat/completions:", oa["choices"][0]["message"]["content"][:120], "...")
print("usage:", oa["usage"])

In [ ]:
# explore: bẫy num_ctx: prompt khoảng 6 000 token, dữ kiện cần nhớ nằm ở ĐẦU prompt
import pyarrow.parquet as pq

WIKI = ensure("wikipedia-vi")
batch = next(pq.ParquetFile(WIKI / "20231101.vi" / "train-00001-of-00004.parquet").iter_batches(batch_size=400, columns=["text"]))
filler, total = [], 0
for t in batch.column("text").to_pylist():
    filler.append(t[:2000])
    total += len(t[:2000])
    if total > 20_000:                     # khoảng 20 000 ký tự tiếng Việt, cỡ 6 000 token Qwen
        break
fact = "Ghi chú quan trọng: mã số của kho lưu trữ là 7429."
long_messages = [{"role": "user", "content": fact + "\n\n" + "\n\n".join(filler)
                  + "\n\nCâu hỏi: theo ghi chú quan trọng ở đầu văn bản, mã số của kho lưu trữ là gì? Chỉ trả lời con số."}]
for num_ctx in (None, 8192):
    body = chat_payload(MODEL, long_messages, stream=False, temperature=0, seed=1, num_predict=16, num_ctx=num_ctx)
    out = http.post("/api/chat", json=body).json()
    s = speed_stats(out)
    print(f"num_ctx={num_ctx or 'mặc định'}: prompt_eval_count {s['prompt_tokens']}, "
          f"trả lời {out['message']['content'].strip()!r}, đúng: {'7429' in out['message']['content']}, "
          f"{s['total_s']:.1f} s")

In [ ]:
# explore: Modelfile: model con "ae09-tro-ly" từ qwen2.5:3b (dùng chung trọng số, chỉ thêm system và tham số), rồi xoá
system = "Bạn là trợ lý tiếng Việt. Luôn trả lời đúng một câu, không quá 25 từ."
params = {"temperature": 0.2, "num_ctx": 4096, "stop": ["<|im_end|>", "<|endoftext|>"]}
print(render_modelfile(MODEL, system, params))
# Cùng nội dung với `ollama create ae09-tro-ly -f Modelfile`, gửi qua REST:
r = http.post("/api/create", json={"model": "ae09-tro-ly", "from": MODEL, "system": system, "parameters": params,
                                   "stream": False})
print("create:", r.json())
ans = http.post("/api/chat", json=chat_payload("ae09-tro-ly", [{"role": "user", "content": "Hugging Face là gì?"}],
                                               stream=False, seed=1)).json()
print("ae09-tro-ly:", ans["message"]["content"])
print("xoá:", http.request("DELETE", "/api/delete", json={"model": "ae09-tro-ly"}).status_code)

## Đọc kết quả

- TTFT so với thời gian sinh cả câu trả lời: người dùng chờ bao lâu mới thấy chữ đầu tiên nếu không stream?
- Tốc độ đọc prompt (`prompt_tps`) và tốc độ sinh (`gen_tps`) chênh nhau bao nhiêu lần? Vì sao?
- `/api/ps` cho thấy model nằm hết trên GPU hay bị chia GPU/CPU? `context_length` đang nạp là bao nhiêu, so với
  context length mà model hỗ trợ trong `/api/show`?
- Với `num_ctx` mặc định, `prompt_eval_count` là bao nhiêu và model có trả lời đúng mã số không? Ollama có báo lỗi
  không?

Sau đó làm phần trắc nghiệm của bài trong app.